# Notebook 05 - Evaluation, Benchmarking, and LLM-as-a-Judge

This notebook computes and interprets quantitative and judge-based metrics.

## What Is This Technique? - Multi-Dimensional Legal Evaluation

### Definition and Core Concepts
Evaluation combines lexical similarity, classification quality, extraction F1, hallucination rates, latency, and LLM-judge scoring.

### Why Was This Technique Developed?
No single metric can capture legal output quality and risk.

### What Limitations of Traditional RAG Does It Solve?
RAG-centric retrieval metrics (Recall@k/MRR) do not measure legal output schema quality. This evaluation fills that generation-quality gap.

### Architecture and Workflow Diagram Explanation
```mermaid
flowchart LR
A[Prediction files] --> B[Metric engine]
A --> C[Hallucination analysis]
A --> D[LLM judges]
B --> E[CSV/JSON reports]
C --> E
D --> E
E --> F[Plots + interpretation]
```

### Component-by-Component Breakdown
1. Summarization: ROUGE, BLEU, METEOR, BERTScore.
2. Risk classification: accuracy/precision/recall/F1.
3. Clause extraction: precision/recall/F1.
4. Hallucination diagnostics.
5. LLM-as-a-judge using Granite and Qwen.

### When Should It Be Used in Real-World Systems?
Use in any production-bound legal AI workflow where reliability matters.

### Advantages and Disadvantages
Advantages:
- Captures multiple failure modes.
- Supports baseline-vs-finetuned comparison.
- Provides operational latency signals.

Disadvantages:
- Judge scores are model-dependent.
- Lexical metrics may miss legal nuance.
- More expensive than single-metric evaluation.

### Comparison Against Standard RAG and Other Implemented Variants
Compared with standard RAG evaluations focused on retrieval relevance, this notebook measures downstream legal generation quality and structure.

### Implementation Details and Design Decisions Used in This Project
Evaluation artifacts are written to `artifacts/metrics/` and `artifacts/figures/`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
ROOT = next((p for p in candidates if (p / 'scripts').exists() and (p / 'configs').exists()), cwd)
print('Project root:', ROOT)

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / script), *args]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT))


In [ ]:
metrics_json = ROOT / 'artifacts/metrics/system_metrics.json'
if not metrics_json.exists():
    run_py('scripts/evaluate.py', '--config', 'configs/default.yaml')
metrics = json.loads(metrics_json.read_text())
metrics.keys()

In [ ]:
metrics_df = pd.read_csv(ROOT / 'artifacts/metrics/system_metrics.csv')
metrics_df.head(20)

In [ ]:
scoreboard_path = ROOT / 'artifacts/metrics/scoreboard.csv'
scoreboard = pd.read_csv(scoreboard_path) if scoreboard_path.exists() else pd.DataFrame()
scoreboard

In [ ]:
latency = json.loads((ROOT / 'artifacts/metrics/latency_report.json').read_text())
pd.DataFrame(latency).T

In [ ]:
for name in ['metric_bars.png', 'risk_confusion_matrix.png', 'system_scoreboard.png']:
    p = ROOT / 'artifacts/figures' / name
    if p.exists():
        plt.figure(figsize=(10, 4))
        plt.imshow(plt.imread(p))
        plt.axis('off')
        plt.title(name)
        plt.show()

In [ ]:
hall = json.loads((ROOT / 'artifacts/metrics/hallucination_report.json').read_text())
judge = json.loads((ROOT / 'artifacts/metrics/judge_report.json').read_text())
pivot = scoreboard.pivot_table(index='system', columns='metric', values='score', aggfunc='mean') if not scoreboard.empty else pd.DataFrame()
best_metrics = pivot.idxmax().to_dict() if not pivot.empty else {}
deltas = {}
if not pivot.empty and 'fine_tuned_adapter' in pivot.index:
    for metric in pivot.columns:
        others = pivot.loc[pivot.index != 'fine_tuned_adapter', metric].dropna()
        if not others.empty:
            deltas[metric] = float(pivot.loc['fine_tuned_adapter', metric] - others.max())

summary = {
    'best_system_per_metric': best_metrics,
    'fine_tuned_delta_vs_best_baseline': deltas,
    'hallucination_report': hall,
    'judge_report': judge,
}
summary

## Post-Run Interpretation and Lessons

1. Read `best_system_per_metric` and `fine_tuned_delta_vs_best_baseline` first to quantify where fine-tuning helped or did not help.
2. Cross-check numeric gains against `hallucination_report` and `judge_report` to detect quality vs faithfulness tradeoffs.
3. Use `latency_report` to decide whether the accuracy gain justifies operational cost.
4. If metrics regress or stall, inspect Notebook 06 row-level errors before retraining.